In [347]:
# Quarterly depedency graph

In [348]:
import os
os.getcwd()

'C:\\Users\\ugne.keliauskaite\\Bruegel\\Research - 2021-11 European natural gas imports\\Data'

In [349]:
#Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data' # Gio
Share_point = r'C:\Users\ugne.keliauskaite\Bruegel\Research - 2021-11 European natural gas imports\Data' # Ugne
os.chdir(Share_point)

In [350]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

In [351]:
# import ENTSOG pipeline data
entsog = pd.read_csv(r'Imports\EU27\df1.csv')
del entsog['dates.1']
entsog = entsog.set_index(pd.DatetimeIndex(entsog['dates']))
del entsog['dates']

In [352]:
# import LNG data from GIE (we only know where the LNG arrives, not where it comes from)
# agsi = pd.read_csv(r'C:\\Users\\giovanni.sgaravatti\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Gio
agsi = pd.read_csv(r'C:\\Users\\ugne.keliauskaite\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Ugne
agsi=agsi.set_index(pd.DatetimeIndex(agsi['dates']))
del agsi['dates']
del agsi['index']

In [353]:
# import Bloomberg LNG data (with these data we know both where it comes from and where it arrives, but we trust GIE better - also to be consistent with the tracker)
lng_b = pd.read_excel(r'LNG\Bloomberg\granular LNG imports.xlsx') # Gio
lng_b.rename(columns= {'Unnamed: 0':'dates'},inplace=True)
lng_b = lng_b.set_index(pd.DatetimeIndex(lng_b['dates']))
del lng_b['dates']

In [354]:
lng_b['Tot'] = lng_b.sum(axis=1,numeric_only=True)

In [355]:
# Divide each column by the 'Tot' column
ratios_df = lng_b.div(lng_b['Tot'], axis=0)

In [356]:
agsi_m = agsi.groupby(pd.Grouper(freq='M'))['sendOut'].sum(numeric_only=True)
# take only values after 2019 to be consistent with Bloomberg data
agsi_19 = agsi_m['2019':]

In [357]:
# to align with Bloombgerg lng
# agsi_19= agsi_19.iloc[:0] #Ugne change this one!

In [358]:
agsi_19

dates
2019-01-31     65357.1
2019-02-28     59014.4
2019-03-31     84049.6
2019-04-30     88014.5
2019-05-31     80457.8
                ...   
2024-12-31    109031.3
2025-01-31    109632.2
2025-02-28    113218.5
2025-03-31    131648.6
2025-04-30     34948.2
Freq: M, Name: sendOut, Length: 76, dtype: float64

In [359]:
# create new dataframe
lng = pd.DataFrame()
lng['dates'] = agsi_19.index

In [360]:
agsi_19
agsi_19= agsi_19.iloc[:-1]

In [361]:
ratios_df= ratios_df.iloc[:-1] #Ugne change this one!

In [362]:
ratios_df

,Qatar,Algeria,Nigeria,Russia,Norway,United States,Trinidad & Tobago,Egypt,Other,Tot
dates,,,,,,,,,,
2019-01-31,0.227813,0.104325,0.187116,0.193431,0.046563,0.139817,0.074051,0.000000,0.026883,1.0
2019-02-28,0.195039,0.117453,0.146219,0.202086,0.062567,0.104679,0.076749,0.000000,0.095208,1.0
2019-03-31,0.217705,0.092288,0.139617,0.207924,0.054209,0.163316,0.068449,0.025039,0.031453,1.0
2019-04-30,0.174857,0.129060,0.138185,0.228259,0.052365,0.155742,0.085473,0.024848,0.011212,1.0
2019-05-31,0.220073,0.115558,0.135534,0.252179,0.054785,0.108252,0.056022,0.024447,0.033150,1.0
...,...,...,...,...,...,...,...,...,...,...
2024-11-30,0.120531,0.082404,0.045933,0.180885,0.053056,0.448179,0.020270,0.000000,0.048742,1.0
2024-12-31,0.140592,0.058965,0.066412,0.216360,0.036114,0.413984,0.009923,0.000000,0.057652,1.0
2025-01-31,0.080595,0.027174,0.073698,0.176758,0.023476,0.562781,0.008509,0.000000,0.047010,1.0


In [363]:
lng = lng.iloc[:-1]

In [364]:
 # multiply Bloomberg LNG ratios by AGSI totals
# and convert to M3m
for column in ratios_df.columns:
    lng[column] = agsi_19.values*ratios_df[column].values/10.3

In [365]:
lng

,dates,Qatar,Algeria,Nigeria,Russia,Norway,United States,Trinidad & Tobago,Egypt,Other,Tot
0,2019-01-31,1445.554676,661.981080,1187.313715,1227.390108,295.455610,887.190554,469.879608,0.000000,170.584164,6345.349515
1,2019-02-28,1117.487058,672.951670,837.772109,1157.860659,358.479163,599.766228,439.738694,0.000000,545.497817,5729.553398
2,2019-03-31,1776.508485,753.087934,1139.297179,1696.694262,442.350493,1332.680771,558.551025,204.325134,256.660058,8160.155340
3,2019-04-30,1494.170233,1102.828446,1180.800058,1950.493874,447.462126,1330.831065,730.372421,212.329440,95.809425,8545.097087
4,2019-05-31,1719.089355,902.671484,1058.714685,1969.877584,427.952750,845.601098,437.613205,190.965978,258.950755,7811.436893
...,...,...,...,...,...,...,...,...,...,...,...
70,2024-11-30,1149.677352,786.008107,438.124364,1725.354486,506.070493,4274.913727,193.344289,0.000000,464.924658,9538.417476
71,2024-12-31,1488.243668,624.177437,703.007851,2290.288141,382.282375,4382.248881,105.039620,0.000000,610.275134,10585.563107
72,2025-01-31,857.841606,289.234934,784.437831,1881.395839,249.875995,5990.187166,90.564407,0.000000,500.365135,10643.902913
73,2025-02-28,1248.075107,410.608391,541.457111,1896.208385,462.674201,5376.713402,100.842176,0.000000,955.508606,10992.087379


In [366]:
lng.set_index(pd.DatetimeIndex(lng['dates']),inplace=True)
del lng['dates']

In [367]:
lng['Total less Russia and USA and NO and AL'] = lng['Tot'] - lng['Russia'] - lng['United States'] - lng['Norway'] - lng['Algeria']

In [368]:
# Change the months for which you have data here
months = pd.date_range(start='2019-01-01', end='2025-03-31', freq='M')

In [369]:
entsog = entsog['2019':]

In [370]:
converter = 10300000   ## KWh to M3m --> 10.3 KWh/m^3     # on ENTSOG/AGSI the data comes in KWh, we transform it (later on) in M3m 

In [371]:
entsog_m = pd.DataFrame()
entsog_m['dates'] = months
entsog_m.set_index(pd.DatetimeIndex(entsog_m['dates']),inplace=True)
del entsog_m['dates']

for country in ['Russia', 'Norway','Algeria', 'UK', 'Azerbaijan','Libya']:
    entsog_m[country] = entsog[entsog['aggregation'] == country]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [372]:
for pipe in ['Ukraine Gas Transit', 'Yamal (BY,PL)','Nord Stream', 'Turkstream']:
    entsog_m[pipe] = entsog[entsog['aggregation2'] == pipe]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [373]:
entsog_q = entsog_m.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)
lng_q = lng.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)

In [374]:
graph = entsog_q
graph['USA LNG'] = lng_q['United States']
graph['Russia LNG'] = lng_q['Russia']
graph['Norway LNG'] = lng_q['Norway']
graph['Algeria LNG'] = lng_q['Algeria']
del graph['Russia']
graph['LNG less RU and USA and NO and AL'] = lng_q['Total less Russia and USA and NO and AL']
graph = graph['2019':]

In [375]:
graph.tail()

,Norway,Algeria,UK,Azerbaijan,Libya,Ukraine Gas Transit,"Yamal (BY,PL)",Nord Stream,Turkstream,USA LNG,Russia LNG,Norway LNG,Algeria LNG,LNG less RU and USA and NO and AL
dates,,,,,,,,,,,,,,
2024-03-31,24152.453088,7446.442823,1869.480192,3202.055103,481.842846,3975.994269,0.0,0.0,3903.919757,16010.149024,5975.658220,1585.975923,2135.291154,5686.333446
2024-06-30,23903.666352,8608.889462,3673.189310,3171.037865,419.845705,4112.429768,0.0,0.0,3874.656347,12858.783785,5074.777019,1419.724317,2780.659912,6207.608364
2024-09-30,21743.384276,7082.813214,5152.462704,2901.612735,212.792361,4178.388680,0.0,0.0,4442.340756,9777.296727,4817.316078,1366.953613,2009.461310,5819.525669
2024-12-31,23481.901287,8859.495027,2043.825797,3389.197710,325.460193,4185.299118,0.0,0.0,4517.038069,12736.733433,5422.692290,1381.890542,2389.187004,7119.457895
2025-03-31,22526.505144,8392.573349,1939.164978,2839.859508,233.619554,0.000000,0.0,0.0,4531.714854,18392.873326,5483.459340,1066.020298,1399.840010,8075.214793


In [376]:
graph = graph[['Nord Stream','Yamal (BY,PL)','Ukraine Gas Transit','Turkstream','Russia LNG', 'Norway LNG', 'USA LNG', 'Algeria LNG', 'LNG less RU and USA and NO and AL', 'Norway','Algeria','UK','Azerbaijan','Libya']]

In [377]:
from datetime import datetime
today = date.today()

In [378]:
with pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today)) as writer:
    Excelwriter = pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today),engine="xlsxwriter")
    entsog_q.to_excel(Excelwriter, sheet_name="ENTSOG", index=True)
    lng.to_excel(Excelwriter, sheet_name="LNG", index=True)
    graph.to_excel(Excelwriter, sheet_name="graph", index=True)
Excelwriter.close()
Excelwriter.save()

C:\Users\ugne.keliauskaite\AppData\Local\Temp\ipykernel_37796\2191511067.py:7: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  Excelwriter.save()
c:\Users\ugne.keliauskaite\AppData\Local\anaconda3\Lib\site-packages\xlsxwriter\workbook.py:368: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")
